## Evaluación de explicabilidad y alertas

**Guía de ejecución:** Requiere corpus, anotaciones, embeddings, splits, checkpoint y predicciones guardadas. Los experimentos SHAP son costosos y requieren preferiblemente GPU.

Instala las dependencias una vez desde la raíz del repositorio con `pip install -e ".[notebooks]"`. Los archivos de `data/` se mantienen fuera de Git. Este notebook se conserva para reproducir la fase de investigación; no se recomienda pulsar «Ejecutar todo» sin revisar sus fases y requisitos.


**Versión adaptada para el repositorio unificado.** Los módulos reutilizables se importan desde `src/bias_detector/`; los datos pesados se guardan bajo `data/` y no se descargan automáticamente. Ejecutar el notebook en orden.

In [ ]:
# Preparación: iniciar Jupyter desde la raíz del repositorio (local o Colab).
from pathlib import Path
import os, sys

def _locate_repo():
    for folder in (Path.cwd(), *Path.cwd().parents, Path('/content/cognitive-bias-detector')):
        if (folder / 'pyproject.toml').is_file() and (folder / 'src' / 'bias_detector').is_dir():
            return folder.resolve()
    raise RuntimeError('No se encontró la raíz del repositorio. Abre Jupyter allí o usa %cd /content/cognitive-bias-detector en Colab.')

PROJECT_ROOT = _locate_repo()
source_path = str(PROJECT_ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
from bias_detector.config import (
    DATA_DIR, MODEL_DATA_DIR, MODEL_DIR, PROMPTS_DIR, RESULTS_DIR,
    RAW_DIR, PROCESSED_DIR, MODELING_DIR, PILOT_DIR,
    REDACTION_ANALYSIS_DIR, ANNOTATION_RESULTS_DIR,
    MASS_ANNOTATION_DIR, FINAL_ANNOTATION_DIR, PILOT_RESULTS_DIR,
    MICRO_PILOT_RESULTS_DIR, CORPUS_DATA, FINAL_ANNOTATIONS_DATA,
    require_file, ensure_output_dir,
)
print('Repositorio:', PROJECT_ROOT)
print('Directorio de datos:', DATA_DIR)


# Explicabilidad SHAP

## Primera parte: identificación de las ventanas responsables de las predicciones del clasificador MIL.

Hay que distinguir que esta identificación de la ventana todavía no constituye una explicación SHAP. Únicamente localiza la instancia que determina el máximo de MIL. SHAP será el método que utilizaremos después para estimar la contribución de los fragmentos de texto.

### Preparación del notebook

In [ ]:
import pandas as pd

In [ ]:
# Artefactos de experimentación por defecto dentro de data/model_data/
BASE_DIR = MODEL_DATA_DIR
SEGMENTATION_DIR = BASE_DIR / 'segmentation'
EMBEDDINGS_DIR = BASE_DIR / 'embeddings'
SPLITS_DIR = BASE_DIR / 'splits'
EXPERIMENT_DIR = MODEL_DIR  # Checkpoint v3 distribuido y umbrales
EVALUATION_DIR = PROJECT_ROOT / 'artifacts' / 'evaluation'
EXPLAINABILITY_DIR = BASE_DIR / 'explainability' / 'mlp_mil_v3' 

In [ ]:
# Dependencias del proyecto: instalar una vez desde la raíz con `%pip install -e ".[notebooks]"`.
%pip install -e ".[notebooks]"

In [ ]:
CORPUS_PATH = CORPUS_DATA
ANNOTATION_PATH = FINAL_ANNOTATIONS_DATA

df_corpus = pd.read_pickle(
    require_file(CORPUS_PATH, "corpus de entrenamiento")
)

df_annotations = pd.read_pickle(
    require_file(ANNOTATION_PATH, "anotaciones finales")
)

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/mil_explainability.py
from bias_detector.mil_explainability import (
    DEFAULT_BIASES, 
    MLPMaxMILClassifier, 
    load_mil_model, 
    compute_window_scores, 
    locate_positive_windows, 
    verify_against_saved_predictions, 
    save_window_attributions
)

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/embedding_extractor.py
from bias_detector.embedding_extractor import (
    KEYS, 
    COLUMNS, 
    save_windows, 
    load_windows, 
    load_encoder, 
    extract_embeddings, 
    load_embeddings
)

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/threshold_optimization.py
from bias_detector.threshold_optimization import (
    ThresholdConfig, 
    metrics_at_thresholds, 
    optimize_thresholds, 
    apply_thresholds, 
    save_thresholds, 
    load_thresholds, 
    plot_precision_recall, 
    plot_confusion_matrices
)

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/segmentation.py
from bias_detector.segmentation import (
    TURN_RE, 
    SegmentationConfig, 
    create_tokenizer, n
    ormalize_conversation, 
    segment_conversation, 
    segment_corpus
)

In [ ]:
# Ventanas y embeddings
df_windows, windows_manifest = load_windows(SEGMENTATION_DIR)

embeddings, window_keys = load_embeddings(EMBEDDINGS_DIR)

# Modelo v3
model, checkpoint = load_mil_model(
    EXPERIMENT_DIR,
    device="cpu",
    biases=DEFAULT_BIASES
)

# Umbrales seleccionados en validación
thresholds, thresholds_metadata = load_thresholds(
    EXPERIMENT_DIR / "thresholds",
    bias_names=DEFAULT_BIASES
)

print("Ventanas:", len(df_windows))
print("Embeddings:", embeddings.shape)
print("Época del modelo:", checkpoint["epoch"])
print("Umbrales:", thresholds)

### Cálculo de las puntuaciones individuales de las ventanas

Hasta ahora nuestro clasificador devolvía una puntuación por sesgo y conversación.

In [ ]:
window_scores = compute_window_scores(
    model=model,
    embeddings=embeddings,
    batch_size=256,
    device="cpu"
)

assert window_scores.shape == (11341, 4)
assert np.isfinite(window_scores).all()

print("Puntuaciones por ventana:", window_scores.shape)

Esto nos proporciona cuatro puntuaciones por cada una de las 11.341 ventanas.

### Identificación de las ventanas dominantes del conjunto de prueba

In [ ]:
# Recuperar los identificadores de test
df_splits = pd.read_csv(
    SPLITS_DIR / "conversation_splits.csv",
    dtype={"conversation_id": str}
)

test_ids = df_splits.loc[
    df_splits["split"] == "test",
    "conversation_id"
].tolist()

# Identificar detecciones y ventanas dominantes
df_conversation_scores, df_positive_windows = (
    locate_positive_windows(
        window_scores=window_scores,
        window_keys=window_keys,
        df_windows=df_windows,
        thresholds=thresholds,
        biases=DEFAULT_BIASES,
        selected_conversation_ids=test_ids
    )
)

print("Conversaciones analizadas:", len(df_conversation_scores))
print("Detecciones positivas:", len(df_positive_windows))

display(
    df_positive_windows[
        [
            "conversation_id",
            "bias",
            "score",
            "threshold",
            "window_id",
            "n_windows",
            "turn_ids"
        ]
    ].head(10)
)

### Verificación de la correspondencia con la evaluación final

In [ ]:
verify_against_saved_predictions(
    conversation_scores=df_conversation_scores,
    predictions_csv=(
        EVALUATION_DIR / "test_predictions.csv"
    ),
    biases=DEFAULT_BIASES,
    atol=1e-5
)

print(
    "Verificación superada: las puntuaciones y "
    "decisiones coinciden con la evaluación final."
)

### Inspección de una detección concreta

In [ ]:
examples = df_positive_windows[
    df_positive_windows["bias"] == "confirmation_bias"
]

example = examples.iloc[0]

print("Conversación:", example["conversation_id"])
print("Sesgo:", example["bias"])
print("Puntuación:", round(example["score"], 4))
print("Umbral:", round(example["threshold"], 4))
print("Ventana:", example["window_id"])
print("Turnos incluidos:", example["turn_ids"])

print("\n=== TEXTO DE LA VENTANA ===\n")
print(example["window_text"])

### Guardado de los resultados después de verificar la correspondencia

In [ ]:
saved_paths = save_window_attributions(
    output_dir=EXPLAINABILITY_DIR / "window_localization",
    conversation_scores=df_conversation_scores,
    positive_windows=df_positive_windows
)

for path in saved_paths:
    print("Guardado:", path)

## Segunda parte: Diseño de las atribuciones SHAP

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/text_shap.py
from bias_detector.text_shap import (
    HEADER, ShapTextConfig, MILTextShapExplainer, save_shap_explanation
)


### Preparar GPU

In [ ]:
# Dependencias: instalar una sola vez con `%pip install -e ".[notebooks]"`.

In [ ]:
import torch
import pandas as pd
import numpy as np

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Ya deben estar disponibles de la etapa anterior:
# model            -> MLP-MIL v3
# df_windows       -> ventanas completas guardadas
# embeddings       -> matriz (11341, 768)
# window_keys      -> identificadores alineados
# df_positive_windows -> detecciones positivas de test

### Crear explicador

In [ ]:
# Configuración pequeña para probar el funcionamiento
shap_config = ShapTextConfig(
    model_name="allenai/longformer-base-4096",
    max_features=8,
    nsamples=64,
    batch_size=1,
    fp16=True,
    seed=42
)

explainer = MILTextShapExplainer(
    mlp=model,
    windows=df_windows,
    embeddings=embeddings,
    window_keys=window_keys,
    config=shap_config,
    device="cuda"
)

print("Explicador preparado correctamente.")

In [ ]:
# Seleccionamos un ejemplo: la primera detección de confirmación
# correspondiente a una sola ventana

candidates = df_positive_windows[
    (df_positive_windows["bias"] == "confirmation_bias") &
    (df_positive_windows["n_windows"] == 1)
]

assert not candidates.empty, "No hay ejemplos con estos criterios."

example = candidates.iloc[0]

conversation_id = str(example["conversation_id"])

print("Conversación:", conversation_id)
print("Sesgo:", example["bias"])
print("Ventana:", example["window_id"])
print("Puntuación original:", example["score"])
print("Umbral:", example["threshold"])

In [ ]:
# Podemos recuperar el texto original
original_text = (
    df_corpus.assign(
        conversation_id=df_corpus["conversation_id"].astype(str)
    )
    .set_index("conversation_id")
    .loc[conversation_id, "conversation_text"]
)

print("Texto original recuperado.")

### Calcular contribuciones SHAP

In [ ]:
df_shap, shap_metadata = explainer.explain(
    conversation_id=conversation_id,
    bias=example["bias"],
    bias_names=DEFAULT_BIASES,
    original_text=original_text,
    winning_window_id=int(example["window_id"])
)

print("=== METADATOS DE LA EXPLICACIÓN ===")

for key, value in shap_metadata.items():
    print(f"{key}: {value}")

print("\n=== CONTRIBUCIONES TEXTUALES ===")

display(
    df_shap[
        [
            "group_id",
            "turn_ids",
            "text",
            "shap_value"
        ]
    ].sort_values(
        "shap_value",
        ascending=False
    )
)

### Validar la explicación

In [ ]:
original_score = float(example["score"])
recomputed_score = shap_metadata["original_score_recomputed"]

print("Puntuación guardada:", round(original_score, 6))
print("Puntuación recalculada:", round(recomputed_score, 6))
print(
    "Diferencia:",
    round(abs(original_score - recomputed_score), 6)
)

print(
    "Error de aditividad SHAP:",
    round(shap_metadata["additivity_error"], 6)
)

# Tolerancia inicial debido a inferencia FP16
assert abs(original_score - recomputed_score) < 0.02

# Tolerancia orientativa para el cálculo SHAP
assert shap_metadata["additivity_error"] < 0.02

print("Validaciones superadas.")


### Visualizar fragmentos relevantes

In [ ]:
import matplotlib.pyplot as plt

def plot_text_shap(df_shap, max_chars=65):
    data = df_shap.sort_values(
        "shap_value",
        ascending=True
    ).copy()

    labels = [
        f"{str(turns)} | {str(text).strip()[:max_chars]}"
        for turns, text in zip(
            data["turn_ids"],
            data["text"]
        )
    ]

    fig, ax = plt.subplots(
        figsize=(11, max(4, len(data) * 0.65))
    )

    ax.barh(
        labels,
        data["shap_value"]
    )

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Contribución SHAP")
    ax.set_title("Atribución textual por fragmento")

    plt.tight_layout()
    return fig


fig = plot_text_shap(df_shap)
plt.show()

In [ ]:
# Guardamos resultados
SHAP_DIR = EXPLAINABILITY_DIR / "text_shap"

saved_paths = save_shap_explanation(
    rows=df_shap,
    metadata=shap_metadata,
    output_dir=SHAP_DIR
)

for path in saved_paths:
    print("Guardado:", path)

### Comparación de SHAP con las anotaciones

In [ ]:
def compare_shap_with_annotations(
    df_shap,
    trigger_turns,
    evidence_turns,
    top_k=3
):
    """
    Compara los turnos presentes en los grupos SHAP
    más relevantes con las anotaciones de referencia.

    Es una comprobación de concordancia, no una
    validación causal de las explicaciones.
    """

    # Seleccionar grupos con contribución positiva
    top_groups = (
        df_shap[df_shap["shap_value"] > 0]
        .sort_values("shap_value", ascending=False)
        .head(top_k)
    )

    # Recuperar los turnos asociados
    highlighted_turns = set()

    for turn_ids in top_groups["turn_ids"]:
        highlighted_turns.update(turn_ids)

    trigger = set(trigger_turns)
    evidence = set(evidence_turns)

    return {
        "highlighted_turns": sorted(highlighted_turns),
        "trigger_turns": sorted(trigger),
        "evidence_turns": sorted(evidence),
        "trigger_overlap": sorted(highlighted_turns & trigger),
        "evidence_overlap": sorted(highlighted_turns & evidence),
        "trigger_recall": (
            len(highlighted_turns & trigger) / len(trigger)
            if trigger else None
        ),
        "evidence_recall": (
            len(highlighted_turns & evidence) / len(evidence)
            if evidence else None
        )
    }

In [ ]:
annotation = df_annotations[
    (df_annotations["conversation_id"].astype(str) == conversation_id) &
    (df_annotations["bias"] == "confirmation_bias")
].iloc[0]

comparison = compare_shap_with_annotations(
    df_shap=df_shap,
    trigger_turns=annotation["final_trigger_turns"],
    evidence_turns=annotation["final_evidence_turns"],
    top_k=3
)

for key, value in comparison.items():
    print(f"{key}: {value}")

In [ ]:
print("Etiqueta real:", annotation["final_label"])

### Evaluación de explicabilidad sobre más ejemplos

Esta etapa servirá para estudiar si las explicaciones son útiles para identificar fragmentos relevantes y, posteriormente, elaborar las alertas del usuario.

Distribución de casos acordada:

| Sesgo | VP | FP |
| :--- | :---: | ---: |
|Confirmacion|4|3|
|Disponibilidad|3|2|
|Total|7|5|

Evaluaremos tres dimensiones:
- Fidelidad técnica: coincidencia entre la puntuación original y la recalculada, y error de aditividad SHAP.
- Localización: concordancia de los turnos destacados con final_trigger_turns y final_evidence_turns, únicamente en verdaderos positivos.
- Coherencia semántica: revisión cualitativa de si los fragmentos destacados parecen relacionados con el mecanismo del sesgo.

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/shap_case_study.py
from bias_detector.shap_case_study import (
    QUOTAS, select_cases, run_case_study, aggregate_case_study, build_qualitative_review
)


In [ ]:
# Seleccionamos los 12 casos
CASE_STUDY_DIR = EXPLAINABILITY_DIR / "shap_case_study"
CASE_STUDY_DIR.mkdir(parents=True, exist_ok=True)

df_cases = select_cases(
    positive_windows=df_positive_windows,
    annotations=df_annotations,
    corpus=df_corpus
)

print("Casos seleccionados:", len(df_cases))

display(
    df_cases[
        [
            "case_id",
            "conversation_id",
            "bias",
            "case_type",
            "n_turns",
            "n_windows",
            "score",
            "final_trigger_turns",
            "final_evidence_turns"
        ]
    ]
)


In [ ]:
display(
    pd.crosstab(
        df_cases["bias"],
        df_cases["case_type"]
    )
)

In [ ]:
# Ejecutamos SHAP de manera controlada
df_shap_summary = run_case_study(
    cases=df_cases,
    corpus=df_corpus,
    explainer=explainer,
    biases=DEFAULT_BIASES,
    output_dir=CASE_STUDY_DIR / "explanations",
    nsamples=64,
    score_tolerance=0.02,
    additivity_tolerance=0.02
)

print("Explicaciones completadas:", len(df_shap_summary))

In [ ]:
# Para conservar la selección
selection_path = CASE_STUDY_DIR / "selected_cases.parquet"

if not selection_path.exists():
    df_cases.to_parquet(selection_path, index=False)
    print("Selección guardada.")
else:
    previous = pd.read_parquet(selection_path)
    assert (
        previous[["conversation_id", "bias"]].reset_index(drop=True)
        .equals(
            df_cases[["conversation_id", "bias"]]
            .reset_index(drop=True)
        )
    )
    print("Selección previamente guardada y verificada.")

In [ ]:
# Evaluamos la localización de los turnos
# Métricas de localización por conversación
cols = [
    "case_id",
    "bias",
    "case_type",
    "n_turns",
    "n_windows",
    "top1_turns",
    "top2_turns",
    "trigger_precision_at_1",
    "trigger_recall_at_1",
    "evidence_recall_at_1",
    "union_precision_at_1",
    "union_recall_at_1",
    "union_precision_at_2",
    "union_recall_at_2"
]

display(
    df_shap_summary[cols].round(4)
)

In [ ]:
# Resumen descriptivo de los siete verdaderos positivos
df_localization_metrics = aggregate_case_study(
    df_shap_summary
)

display(df_localization_metrics.round(4))

In [ ]:
# Examinación de la fidelidad de las explicaciones
display(
    df_shap_summary[
        [
            "case_id",
            "bias",
            "case_type",
            "score",
            "original_score_recomputed",
            "additivity_error",
            "n_groups"
        ]
    ].round(6)
)

print(
    "Error máximo de aditividad:",
    df_shap_summary["additivity_error"].max()
)

print(
    "Diferencia máxima de puntuaciones:",
    (
        df_shap_summary["score"]
        - df_shap_summary["original_score_recomputed"]
    ).abs().max()
)

In [ ]:
# Resivisón cualitativa de los fragmentos
df_qualitative = build_qualitative_review(
    cases=df_cases,
    shap_dir=CASE_STUDY_DIR / "explanations",
    top_n=3
)

review_path = (
    CASE_STUDY_DIR / "qualitative_review.csv"
)

if not review_path.exists():
    df_qualitative.to_csv(
        review_path,
        index=False
    )

display(
    df_qualitative[
        [
            "case_id",
            "bias",
            "case_type",
            "top_fragments"
        ]
    ]
)

En resumen: la explicabilidad es técnicamente viable y está validada numéricamente, pero su capacidad de localización es parcial. Esta conclusión no impide desarrollar el prototipo de alertas; nos indica cómo deben formularse para evitar transmitir una certeza que el modelo no puede garantizar.

## Módulo de generación de alertas explicables

Último componente funcional de nuestro sistema de detección de sesgos cognitivos.

El módulo tiene como finalidad transformar las predicciones del clasificador en mensajes comprensibles que inviten al usuario a revisar críticamente determinados aspectos de una conversación.

El sistema seguirá siendo retrospectivo: analizará conversaciones ya finalizadas y mostrará las alertas después del análisis. Su posible funcionamiento en tiempo real quedará como línea de desarrollo futuro.

El módulo no pretende certificar que exista manipulación cognitiva. Su función es señalar posibles patrones que convendría examinar.

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/alert_generator.py
from bias_detector.alert_generator import (
    BIASES, TEMPLATES, DISCLAIMER, MissingExplanationError, Alert, generate_alerts
)


In [ ]:
# Ejemplo mínimo de uso
scores = {
    "anchoring_bias": 0.05,
    "availability_bias": 0.08,
    "confirmation_bias": 0.435799,
    "framing_effect": 0.10
}

thresholds = {
    "anchoring_bias": 0.1701,
    "availability_bias": 0.1233,
    "confirmation_bias": 0.3226,
    "framing_effect": 0.2407
}

shap_fragments = {
    "confirmation_bias": [
        {
            "group_id": 1,
            "turn_ids": [2],
            "text": (
                "Yes, abbreviating it truthfully suggests "
                "that journalism accounts for and debunks "
                "male supremacy thought."
            ),
            "shap_value": 0.083166
        }
    ]
}

alerts = generate_alerts(
    conversation_id="CORPUS_01511",
    scores=scores,
    thresholds=thresholds,
    shap_fragments=shap_fragments
)

import json

print(
    json.dumps(
        alerts,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# Para algún caso en el que ya hayamos calculado SHAP

# Recuperar las puntuaciones reales de la conversación
conversation_id = "CORPUS_01511"

row = df_conversation_scores.loc[
    df_conversation_scores["conversation_id"].astype(str)
    == conversation_id
].iloc[0]

# Utilizar las columnas de puntuación que correspondan
# a cada sesgo en df_conversation_scores.
scores = {
    bias: float(row[f"{bias}_score"])
    for bias in BIASES
}

# Umbrales definitivos, sin redondear
threshold_dict = {
    bias: float(thresholds[bias])
    for bias in BIASES
}

# Adaptar la tabla SHAP al formato del generador
fragments = df_shap[
    ["group_id", "turn_ids", "text", "shap_value"]
].to_dict(orient="records")

shap_fragments = {
    "confirmation_bias": fragments
}

alerts = generate_alerts(
    conversation_id=conversation_id,
    scores=scores,
    thresholds=threshold_dict,
    shap_fragments=shap_fragments
)

## Arquitectura end-to-end (pasar a otro notebook)

In [ ]:
# Implementación compartida: código fuente en src/bias_detector/pipeline.py
from bias_detector.pipeline import (
    PipelineConfig, BiasDetectionPipeline, save_pipeline_result
)


In [ ]:
PIPELINE_RESULTS_DIR = (
    BASE_DIR / "pipeline_results"
)

PIPELINE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# Inicializar el pipeline
pipeline_config = PipelineConfig(
    model_name="allenai/longformer-base-4096",
    max_length=2048,
    overlap=128,
    shap_max_features=8,
    shap_nsamples=64,
    shap_batch_size=1,
    embedding_batch_size=1,
    use_fp16=True,
    score_tolerance=0.02,
    shap_additivity_tolerance=0.02
)

pipeline = BiasDetectionPipeline(
    experiment_dir=EXPERIMENT_DIR,
    config=pipeline_config,
    device="cuda"
)

print("Pipeline inicializado.")
print("Umbrales definitivos:", pipeline.thresholds)

In [ ]:
# Ejecutar pipeline sobre una conversación real
conversation_id = "CORPUS_01511"

conversation_text = (
    df_corpus.assign(
        conversation_id=df_corpus["conversation_id"].astype(str)
    )
    .set_index("conversation_id")
    .loc[conversation_id, "conversation_text"]
)

result = pipeline.run(
    conversation_id=conversation_id,
    conversation_text=conversation_text
)

print("Estado:", result["status"])
print("Ventanas:", result["n_windows"])
print("Sesgos detectados:", result["detected_biases"])
print("Alertas:", len(result["alerts"]))

In [ ]:
# Examinar puntuaciones
df_result_scores = pd.DataFrame([
    {
        "bias": bias,
        "score": result["scores"][bias],
        "threshold": result["thresholds"][bias],
        "detected": bias in result["detected_biases"]
    }
    for bias in pipeline.thresholds
])

display(df_result_scores.round(6))

In [ ]:
# Visualizar alertas

for alert in result["alerts"]:

    print("=" * 75)
    print(alert["title"])
    print("=" * 75)

    print("\nDescripción:")
    print(alert["summary"])

    print("\nPuntuación:", round(alert["score"], 6))
    print("Umbral:", round(alert["threshold"], 6))

    print(
        "\nFragmentos influyentes:",
        alert["n_shap_fragments"]
    )

    for fragment in alert["shap_fragments"]:
        print("\nTurnos:", fragment["turn_ids"])
        print("SHAP:", round(fragment["shap_value"], 6))
        print(fragment["text"])

    print("\nRecomendación:")
    print(alert["review_question"])

    print("\nLimitaciones:")
    print(alert["disclaimer"])

In [ ]:
# Guardamos salida estructurada
output_path = (
    PIPELINE_RESULTS_DIR
    / f"{conversation_id}_alerts.json"
)

saved_path = save_pipeline_result(
    result,
    output_path
)

print("Resultado guardado:", saved_path)

In [ ]:
# Validación funcional extremo a extremo

# from alert_generator import BIASES

# 1. Los umbrales deben ser los definitivos.
assert set(result["scores"]) == set(BIASES)
assert set(result["thresholds"]) == set(BIASES)

# 2. Las detecciones deben corresponder a los umbrales.
expected_detected = [
    bias for bias in BIASES
    if result["scores"][bias] >= result["thresholds"][bias]
]

assert result["detected_biases"] == expected_detected

# 3. Debe existir una alerta por detección.
assert len(result["alerts"]) == len(expected_detected)

# 4. Todas las alertas deben contener SHAP.
for alert in result["alerts"]:
    fragments = alert["shap_fragments"]

    assert len(fragments) > 0
    assert len(fragments) == alert["n_shap_fragments"]

    assert all(
        fragment["shap_value"] > 0
        for fragment in fragments
    )

    # No debe haber fragmentos truncados por el generador.
    assert all(
        isinstance(fragment["text"], str)
        and fragment["text"].strip()
        for fragment in fragments
    )

    # Las atribuciones deben estar ordenadas.
    values = [
        fragment["shap_value"]
        for fragment in fragments
    ]

    assert values == sorted(values, reverse=True)

# 5. Validaciones numéricas de SHAP.
for bias, metadata in result["explanations"].items():

    assert metadata["score_difference"] <= 0.02

    assert metadata["additivity_error"] <= 0.02

print("Todas las validaciones funcionales superadas.")

Hay tres limitaciones que conservaríamos expresamente en la documentación: SHAP solo explica la ventana dominante, una atribución positiva no certifica la existencia del sesgo y el coste computacional del proceso completo todavía no ha sido evaluado para un despliegue productivo.

### Últimas validaciones de la parte técnica

In [ ]:
# Conversaciones sin detecciones positivas

from unittest.mock import patch

# Simular puntuaciones inferiores a todos los umbrales
def mock_negative_scores(*args, **kwargs):
    embeddings = args[1]
    n_windows = len(embeddings)

    return np.zeros(
        (n_windows, len(BIASES)),
        dtype=np.float32
    )


# Obtener el espacio global donde se definió el pipeline
pipeline_globals = BiasDetectionPipeline.run.__globals__

# Guardar referencias originales
original_compute = pipeline_globals["compute_window_scores"]
original_shap = pipeline_globals["MILTextShapExplainer"]

from unittest.mock import MagicMock

mock_shap = MagicMock()

try:
    # Sustituir temporalmente las funciones
    pipeline_globals["compute_window_scores"] = mock_negative_scores
    pipeline_globals["MILTextShapExplainer"] = mock_shap

    # Ejecutar pipeline con puntuaciones simuladas
    result_negative = pipeline.run(
        conversation_id="TEST_NO_ALERTS",
        conversation_text=conversation_text
    )

    # Comprobar que SHAP no se ha inicializado
    mock_shap.assert_not_called()

finally:
    # Restaurar SIEMPRE las referencias originales
    pipeline_globals["compute_window_scores"] = original_compute
    pipeline_globals["MILTextShapExplainer"] = original_shap


# Validaciones funcionales
assert result_negative["status"] == "no_alerts"
assert result_negative["alerts"] == []
assert result_negative["detected_biases"] == []
assert result_negative["explanations"] == {}

assert all(
    value == 0.0
    for value in result_negative["scores"].values()
)

print("PRUEBA A SUPERADA")
print("Estado:", result_negative["status"])
print("Alertas:", len(result_negative["alerts"]))
print("SHAP ejecutado: No")

In [ ]:
# Fallo durante la generación de SHAP

import numpy as np
from unittest.mock import MagicMock

# --------------------------------------------------
# 1. Simular una detección positiva de confirmación
# --------------------------------------------------

def mock_positive_scores(*args, **kwargs):
    embeddings = args[1]
    n_windows = len(embeddings)

    scores = np.zeros(
        (n_windows, len(BIASES)),
        dtype=np.float32
    )

    confirmation_index = list(BIASES).index(
        "confirmation_bias"
    )

    scores[:, confirmation_index] = 0.90

    return scores


# --------------------------------------------------
# 2. Simular un fallo en SHAP
# --------------------------------------------------

fake_explainer = MagicMock()

fake_explainer.explain.side_effect = RuntimeError(
    "Fallo SHAP simulado para prueba funcional"
)


# --------------------------------------------------
# 3. Acceder al espacio global del pipeline
# --------------------------------------------------

pipeline_globals = BiasDetectionPipeline.run.__globals__

# Guardar las referencias originales
original_compute = pipeline_globals["compute_window_scores"]
original_shap = pipeline_globals["MILTextShapExplainer"]
original_generate = pipeline_globals["generate_alerts"]

# Mock del generador de alertas
mock_generate_alerts = MagicMock()

caught_error = None
result_failure = None


# --------------------------------------------------
# 4. Ejecutar la prueba controlada
# --------------------------------------------------

try:

    # Sustituir temporalmente las funciones
    pipeline_globals["compute_window_scores"] = mock_positive_scores

    pipeline_globals["MILTextShapExplainer"] = (
        MagicMock(return_value=fake_explainer)
    )

    pipeline_globals["generate_alerts"] = mock_generate_alerts

    # Ejecutar el pipeline
    try:
        result_failure = pipeline.run(
            conversation_id="TEST_SHAP_FAILURE",
            conversation_text=conversation_text
        )

    except RuntimeError as exc:
        caught_error = exc

    # SHAP debe haberse intentado ejecutar
    fake_explainer.explain.assert_called_once()

    # No debe haberse invocado el generador de alertas
    mock_generate_alerts.assert_not_called()

finally:

    # Restaurar SIEMPRE las referencias originales
    pipeline_globals["compute_window_scores"] = original_compute

    pipeline_globals["MILTextShapExplainer"] = original_shap

    pipeline_globals["generate_alerts"] = original_generate


# --------------------------------------------------
# 5. Validaciones funcionales
# --------------------------------------------------

assert caught_error is not None, (
    "El pipeline no ha propagado el error de SHAP."
)

assert "Fallo SHAP simulado" in str(caught_error)

assert result_failure is None, (
    "El pipeline ha devuelto un resultado parcial."
)

print("PRUEBA B SUPERADA")
print("Excepción capturada:", caught_error)
print("Generador de alertas ejecutado: No")
print("Resultado parcial devuelto: No")

In [ ]:
# Persistencia JSON

import json
import tempfile
from pathlib import Path

# --------------------------------------------------
# 1. Verificar que existe un resultado real
# --------------------------------------------------

assert result["conversation_id"] == "CORPUS_01511"

assert result["status"] == "alerts_generated"

assert len(result["alerts"]) > 0


# --------------------------------------------------
# 2. Crear un directorio temporal para la prueba
# --------------------------------------------------

with tempfile.TemporaryDirectory() as temp_dir:

    json_path = Path(temp_dir) / "pipeline_test.json"

    # --------------------------------------------------
    # 3. Guardar el resultado utilizando nuestra función
    # --------------------------------------------------

    saved_path = save_pipeline_result(
        result=result,
        destination=json_path
    )

    # Comprobar que el archivo existe
    assert saved_path.exists()

    # --------------------------------------------------
    # 4. Recuperar el resultado almacenado
    # --------------------------------------------------

    with saved_path.open(
        "r",
        encoding="utf-8"
    ) as file:

        recovered = json.load(file)

    # --------------------------------------------------
    # 5. Verificar la integridad de los datos
    # --------------------------------------------------

    # El JSON recuperado debe coincidir exactamente
    # con el resultado original del pipeline.
    assert recovered == result, (
        "El resultado recuperado no coincide con el original."
    )

    # Identificador de conversación
    assert recovered["conversation_id"] == "CORPUS_01511"

    # Estado del procesamiento
    assert recovered["status"] == "alerts_generated"

    # Número de alertas
    assert len(recovered["alerts"]) == len(result["alerts"])

    # Puntuaciones y umbrales
    assert recovered["scores"] == result["scores"]
    assert recovered["thresholds"] == result["thresholds"]

    # Explicaciones SHAP
    assert recovered["explanations"] == result["explanations"]

    # Comprobar fragmentos de cada alerta
    for original_alert, recovered_alert in zip(
        result["alerts"],
        recovered["alerts"]
    ):

        assert (
            recovered_alert["bias"]
            == original_alert["bias"]
        )

        assert (
            recovered_alert["shap_fragments"]
            == original_alert["shap_fragments"]
        )

        assert (
            recovered_alert["n_shap_fragments"]
            == original_alert["n_shap_fragments"]
        )

    print("PRUEBA C SUPERADA")
    print("Conversación:", recovered["conversation_id"])
    print("Estado:", recovered["status"])
    print("Alertas recuperadas:", len(recovered["alerts"]))
    print("Explicaciones recuperadas:", len(recovered["explanations"]))
    print("Integridad JSON: Correcta")